# 昆虫分类问题：k-近邻（kNN）分类模型求解设计

## 一、k-近邻（kNN）分类模型组成与核心参数
### 1. 模型组成
- 特征矩阵：昆虫的触长、翅长两个数值型特征；
- 标签向量：昆虫类别（A 类、B 类）；
- 距离度量：欧式距离（适用于低维连续特征，计算样本间相似性）；
- 核心参数：`n_neighbors`（k 值，近邻个数）。

### 2. 参数确定方式
- 数据集规模：共 12 个样本（A 类 6 个、B 类 6 个），属于小样本数据集，选择经验最优 k 值（k=3，奇数可避免投票平局）；
- 特征预处理：触长和翅长数值量级一致（均为 1.x），无需标准化，直接计算距离。

### 3. 新样本预测流程
1. 构建训练数据集（特征 + 标签）；
2. 计算新样本（触长 1.2、翅长 1.8）与所有训练样本的欧式距离；
3. 选取距离最小的 k 个近邻；
4. 多数投票法确定新样本类别（k 个近邻中占比最高的类别）。

## 二、技术路线
1. 手动构建昆虫数据集（特征矩阵 + 标签向量）；
2. 定义新预测样本；
3. 计算新样本与训练样本的欧式距离；
4. 筛选 k=3 个最近邻，统计类别分布；
5. 投票确定预测类别并输出结果。

# Rice-YOLOv11n: 基于无人机视角的水稻测产系统

本项目是一款针对无人机视角下水稻测产设计的轻量化目标检测系统。基于5种基准算法选出最优的YOLOv11n算法再通过改进 YOLOv11n 架构，引入 **P2 小目标检测头** 和 **EMA 注意力机制**以及引入**MPDIoU**损失函数，得到Rice-YOLOv11n模型实现了对高空视角下微小、密集且重叠的稻穗目标的高精度识别。

---

##  实验环境配置

本实验所有模型训练与推理验证均在以下硬件及环境下完成（AutoDL算力云平台）：

| 环境类别 | 配置参数 |
| :--- | :--- |
| **操作系统** | Ubuntu 22.04 LTS |
| **显卡 (GPU)** | NVIDIA GeForce RTX 2080 Ti (11GB) |
| **处理器 (CPU)** | Intel(R) Xeon(R) Silver 4214R @ 2.40GHz |
| **开发框架** | Python 3.10 / PyTorch 2.1.0 / Ultralytics 8.3 |
| **内存 (RAM)** | 32GB |
| **加速环境** | CUDA 12.1 / cuDNN 8.9.2 |

---

##  目录结构说明

根据项目根目录文件清单，各核心文件功能如下：

*   **`nn/`**: 存放改进模型的神经网络核心模块代码。
*   **`yolov11-rice-p2-ema.yaml`**: 改进后的 Rice-YOLOv11n 模型配置文件（含 P2 层与 EMA 模块）。
*   **`data.yaml`**: 数据集路径配置文件。
*   **`UI.py`/ `fafu_bg.jpg`**: 基于 PyQt5 开发的可视化操作界面脚本及资源文件。
*   **`train_advanced.py`**: 改进算法（Rice-YOLOv11n）的训练启动脚本。
*   **`train_base.py`**: YOLO 系列基准算法（v8/v10/v11）和RT-DETR的对比训练脚本。
*   **`train_base_rcnn.py`**: Faster R-CNN 算法的专用训练脚本。
*   **`test.py`**: YOLO 系列及 RT-DETR 的模型测试与指标评估脚本。
*   **`test_rcnn.py`**: Faster R-CNN 专用的验证与推理脚本。
*   **`MainActivity.java`**: 用于 Android 端部署的相关源代码。

---

##  运行方式

### 1. 环境准备

pip install ultralytics==8.3.0 torch==2.1.0 torchvision --index-url [https://download.pytorch.org/whl/cu121](https://download.pytorch.org/whl/cu121)

pip install PyQt5 pandas matplotlib opencv-python

pip install scipy torchmetrics

### 2. 模型训练
训练改进算法: 运行 python train_advanced.py

训练基准模型: 运行 python train_base.py

训练 Faster R-CNN: 运行 python train_base_rcnn.py

YOLO 系列、RT-DETR 测试评估：运行 python test.py

Faster R-CNN 模型测试：运行 python test_rcnn.py

### 3.可视化界面运行
python UI.py

### 4.注意事项

路径匹配: 运行前请确保 data.yaml 中的 train/val/test 路径指向 AutoDL 上的实际绝对路径。

显存管理: RTX 2080 Ti 显存为 11GB，跑 train_base_rcnn.py 时建议 batch_size 设为 4 以防溢出。

### 维护者: 3238507069王智颍

In [1]:
# 昆虫分类问题 - k-近邻（kNN）分类预测
import numpy as np
# ===================== 1. 构建训练数据集 =====================
# A类昆虫特征（触长、翅长）：6个样本
a_features = np.array([
    [1.1, 1.8],
    [1.1, 1.9],
    [1.2, 1.9],
    [1.2, 2.0],
    [1.3, 2.2],
    [1.3, 2.3]
])
# B类昆虫特征（触长、翅长）：6个样本
b_features = np.array([
    [1.3, 1.6],
    [1.3, 1.5],
    [1.4, 1.6],
    [1.4, 1.7],
    [1.5, 1.6],
    [1.5, 1.7]
])
# 合并特征矩阵：X (12, 2)
X = np.vstack((a_features, b_features))
# 构建标签向量：A类标记为0，B类标记为1 (12,)
y = np.array([0]*6 + [1]*6)
# 类别名称映射（便于输出解读）
class_names = {0: "A类昆虫", 1: "B类昆虫"}

# ===================== 2. 定义待预测新样本 =====================
new_sample = np.array([1.2, 1.8])  # 触长1.2、翅长1.8

# ===================== 3. 计算欧式距离 =====================
# 向量化计算：新样本与所有训练样本的欧式距离
# 步骤：差值 → 平方 → 行求和 → 开根号
distances = np.sqrt(np.sum((X - new_sample) ** 2, axis=1))

# ===================== 4. 筛选k=3个最近邻 =====================
k = 3
# 按距离升序排序，取前k个索引
nearest_indices = np.argsort(distances)[:k]
# 获取近邻的标签
nearest_labels = y[nearest_indices]
# 获取近邻的特征（可选，用于输出查看）
nearest_samples = X[nearest_indices]

# ===================== 5. 多数投票确定预测类别 =====================
# 统计各类别出现次数
unique_labels, counts = np.unique(nearest_labels, return_counts=True)
# 选择出现次数最多的类别
pred_label = unique_labels[np.argmax(counts)]
pred_class = class_names[pred_label]

# ===================== 6. 输出结果 =====================
print("="*50)
print("昆虫分类kNN预测结果")
print("="*50)
print(f"待预测样本：触长={new_sample[0]}, 翅长={new_sample[1]}")
print(f"选择k={k}个最近邻，近邻索引：{nearest_indices}")
print(f"近邻样本特征：\n{nearest_samples}")
print(f"近邻样本类别标签：{nearest_labels} → 对应类别：{[class_names[l] for l in nearest_labels]}")
print(f"类别投票统计：{dict(zip([class_names[l] for l in unique_labels], counts))}")
print(f"\n最终预测类别：{pred_class}")
print("="*50)

昆虫分类kNN预测结果
待预测样本：触长=1.2, 翅长=1.8
选择k=3个最近邻，近邻索引：[0 2 1]
近邻样本特征：
[[1.1 1.8]
 [1.2 1.9]
 [1.1 1.9]]
近邻样本类别标签：[0 0 0] → 对应类别：['A类昆虫', 'A类昆虫', 'A类昆虫']
类别投票统计：{'A类昆虫': np.int64(3)}

最终预测类别：A类昆虫


## 分类模型及预测结果
### 一、分类模型信息
#### 1. 模型核心配置
- **特征维度**：昆虫的触长、翅长两个连续型特征，无特征预处理（特征量级一致，直接使用原始数据）；
- **距离度量**：采用欧式距离计算样本间相似性；
- **核心参数**：`n_neighbors=3`（选择3个最近邻样本）；
- **分类规则**：多数投票法（近邻中占比最高的类别为预测类别）；
- **类别映射**：0代表A类昆虫，1代表B类昆虫。

#### 2. 训练数据集
- A类昆虫（6个样本）：(1.1,1.8)、(1.1,1.9)、(1.2,1.9)、(1.2,2.0)、(1.3,2.2)、(1.3,2.3)；
- B类昆虫（6个样本）：(1.3,1.6)、(1.3,1.5)、(1.4,1.6)、(1.4,1.7)、(1.5,1.6)、(1.5,1.7)。

### 二、预测结果
#### 1. 待预测样本
触长=1.2，翅长=1.8。

#### 2. 近邻分析结果
- 3个最近邻索引：[0, 2, 1]；
- 近邻样本特征：[[1.1, 1.8], [1.2, 1.9], [1.1, 1.9]]；
- 近邻类别标签：[0, 0, 0]（均为A类昆虫）；
- 类别投票统计：A类昆虫获得3票，B类昆虫0票。

#### 3. 最终预测结论
触长1.2、翅长1.8的昆虫被预测为 **A类昆虫**，预测结果无投票分歧，类别判定明确。